In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
class PositionalEncoding(nn.Module):
    """
    Create a Positional Encoding Layer
    """

    def __init__(self):
        super().__init__()

    # Recieve a 2d matrix
    def forward(self, input_tensor: torch.tensor) -> torch.tensor:
        self.num_of_tokens = input_tensor.shape[1]
        self.d_model = input_tensor.shape[2]
        vector1 = torch.arange(self.num_of_tokens).reshape(-1, 1).float()
        # Vector from 0 to n-1 Shape(n, 1)

        vector2 = torch.arange(self.d_model/2).reshape(-1, 1)
        vector3 = torch.e**(-2*vector2*math.log(10000)/self.d_model).T
        # Vector for division terms Shape(1, d_model)

        matrix = torch.matmul(vector1, vector3)
        # Matrix with shape(num_of_tokens, d_model/2)

        cosine_matrix = torch.cos(matrix)
        sine_matrix = torch.sin(matrix)
        
        # 1. Create an empty matrix of shape (num_of_tokens, d_model)
        pe_matrix = torch.zeros(self.num_of_tokens, self.d_model)

        # 2. Fill the even columns (0, 2, 4...) with sine_matrix
        pe_matrix[:, 0::2] = sine_matrix

        # 3. Fill the odd columns (1, 3, 5...) with cosine_matrix
        pe_matrix[:, 1::2] = cosine_matrix
        
        output_tensor = input_tensor + pe_matrix

        return output_tensor

In [3]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [4]:
class FFNN(nn.Module):
    """
    Apply position-wise feed-forward network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.hidden_layer = nn.Linear(d_model, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, d_model)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        if x.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {x.shape[-1]}")

        z1 = self.hidden_layer(x)
        a1 = self.activation(z1)
        a1 = self.dropout(a1) # Randomly zeroes out some neurons to prevent overfitting
        output = self.output_layer(a1)

        return output

In [5]:
class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim 
        self.attention_layer = MultiHeadAttention(self.d_model, self.num_heads)
        self.ffnn = FFNN(self.d_model, self.hidden_dim, dropout_rate=dropout_rate)
        self.norm1= nn.LayerNorm(self.d_model)
        self.norm2 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: Ouput of Positional Encoding
        # Dimension: (batch_size, num_tokens, d_model)
    
        # ----ATTENTION----
        # Applying Pre-Norm: Normalizing Before Residual Connection
        attn_input = self.norm1(x)
        # Normalizing First

        attn_output = self.attention_layer(attn_input)
        # Passing though attention layer
        
        x = x + self.dropout(attn_output)
        # Applying Residual Connection

        # ----FFNN----
        ffnn_input = self.norm2(x)
        # Normalizing
        ffnn_output = self.ffnn(ffnn_input)
        # Passing through network
        output = x + self.dropout(ffnn_output)
        # Applying Residual Connection
        
        # Dimension remains same: (batch_size, num_tokens, d_model)
        return output


In [6]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
hidden_dim = 16

In [7]:
dummy_block_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_block_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_block_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.0664, 0.7207, 0.2968, 0.9248, 0.4110, 0.7449, 0.1530, 0.3296],
         [0.1379, 0.2847, 0.7604, 0.3008, 0.7064, 0.1530, 0.7336, 0.7003],
         [0.2825, 0.0759, 0.1870, 0.4222, 0.6971, 0.7630, 0.6419, 0.4275]],

        [[0.7963, 0.6043, 0.0432, 0.2579, 0.3917, 0.9089, 0.5120, 0.0532],
         [0.1720, 0.9777, 0.1314, 0.7863, 0.0843, 0.4497, 0.4245, 0.2284],
         [0.2406, 0.9522, 0.4043, 0.8156, 0.1846, 0.5502, 0.8022, 0.9877]]])

In [8]:
encoder = EncoderBlock(d_model, num_heads, hidden_dim)
encoder

EncoderBlock(
  (attention_layer): MultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=False)
    (w_k): Linear(in_features=8, out_features=8, bias=False)
    (w_v): Linear(in_features=8, out_features=8, bias=False)
    (w_o): Linear(in_features=8, out_features=8, bias=False)
  )
  (ffnn): FFNN(
    (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
    (output_layer): Linear(in_features=16, out_features=8, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [9]:
dummy_output = encoder(dummy_block_input)
dummy_output

tensor([[[-0.8392,  0.6660,  0.3215,  0.9654,  0.5903,  0.5636, -0.0215,
           0.9653],
         [-0.3149,  0.0629,  0.5189,  0.1788,  1.1814,  0.0293,  0.4716,
           1.0416],
         [ 0.1234,  0.3085,  0.1090,  0.3527,  1.1793,  0.4922,  0.4650,
           0.9286]],

        [[ 0.1802,  1.1857,  0.3859,  0.2183,  0.6128,  1.5446,  0.7595,
           0.4078],
         [-0.6945,  1.5104,  0.7157,  0.5600,  0.0042,  1.0231,  0.7297,
           0.8310],
         [-0.6028,  1.0939,  0.9942,  0.5180, -0.3011,  0.6294,  0.8034,
           1.2547]]], grad_fn=<AddBackward0>)

In [10]:
# Now, Stack Multiple Encoders:
class EncoderStack(nn.Module):
    
    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, hidden_dim) for _ in range(num_blocks)])
        self.norm_1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of x: (batch_size, num_tokens, d_model) 

        for layer in self.layers:
            x = layer(x)
        output = self.norm_1(x)
        # Dimension: (Batch_size, num_tokens, d_model)

        return output

In [11]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 5
hidden_dim = 16
num_blocks = 6

In [16]:
dummy_stack_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_stack_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_stack_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.5365, 0.2405, 0.4296, 0.3335, 0.1126, 0.4241, 0.1175, 0.8518],
         [0.9057, 0.8031, 0.7702, 0.4545, 0.6739, 0.4863, 0.7798, 0.3354],
         [0.5931, 0.3936, 0.7979, 0.1430, 0.9002, 0.0950, 0.8002, 0.8702]],

        [[0.3461, 0.9480, 0.3616, 0.9546, 0.8225, 0.6805, 0.8157, 0.5241],
         [0.3256, 0.4605, 0.3788, 0.9668, 0.6171, 0.8902, 0.1722, 0.8207],
         [0.9035, 0.6868, 0.0703, 0.0809, 0.0373, 0.2436, 0.1118, 0.3623]]])

In [17]:
encoder_stack = EncoderStack(d_model, num_heads, hidden_dim, num_blocks)
encoder_stack

EncoderStack(
  (layers): ModuleList(
    (0-5): 6 x EncoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
        (output_layer): Linear(in_features=16, out_features=8, bias=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm_1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [18]:
encoder_stack_output = encoder_stack(dummy_stack_input)
print(encoder_stack_output.shape)
encoder_stack_output

torch.Size([2, 3, 8])


tensor([[[ 0.4248, -0.4463,  1.3425,  0.3757, -0.1673, -2.3102,  0.3303,
           0.4504],
         [ 1.3630, -0.4364,  1.3740,  0.1615,  0.0202, -1.9768, -0.2816,
          -0.2238],
         [ 0.1133, -0.4809,  1.6331, -0.0188,  0.3624, -2.1759,  0.1076,
           0.4592]],

        [[-0.3945,  0.1173, -0.1792,  1.1131,  1.1191, -2.1991,  0.6484,
          -0.2250],
         [-0.0778, -0.4288, -0.4488,  1.1399,  1.5635, -1.9488,  0.2530,
          -0.0521],
         [ 1.3318,  0.9397, -1.5712, -0.3848,  0.6375, -1.4732,  0.1753,
           0.3450]]], grad_fn=<NativeLayerNormBackward0>)